# Superseded prototype: old Slice V1 phase-0 baseline

> Do not use this notebook for the current V1. ADR-0009 replaces this
> Qwen3-30B-A3B/Ollama/GGUF experiment with the pinned Duck Qwen 3.6 27B
> FP8/vLLM reproduction in `docs/SLICES.md`. Retained for provenance.

Confronts the riskiest unknown in `docs/PLAN.md`: does a quantized ~27-30B model
even run inside Kaggle's constraints at all, and how does it compare to the
naive existing baseline (`arc-agi-3-benchmarking`)?

Build plan (see `docs/SLICES.md` V1):

1. Convert / download Qwen3-30B-A3B as GGUF (ADR-0007) and confirm llama.cpp
   loads and runs it -- the single highest-risk step in the project. If this
   fails, stop and revisit ADR-0002/ADR-0007 before anything else.
2. Quantize to 4-bit, measure tokens/sec on this session's assigned GPU.
3. Wire the quantized model into `arc-agi-3-benchmarking`'s existing naive
   loop via an OpenAI-compatible local server (see
   `configs/model_configs.local.yaml`).
4. Record RHAE and tokens/sec.

**Run this notebook once per GPU option** (T4x2, P100, RTX 6000 Ada) --
Kaggle assigns the accelerator per session, so a single run only measures one
option. Save each run's summary table (last cell) back into this repo under
`docs/research/phase0-results/`.

This notebook needs internet (to download the model and to reach
arcprize.org's public API for game data) -- that's fine, this is dev-time
benchmarking, not the final no-internet Kaggle submission (ADR-0004).


## 0. Environment check

Confirm which GPU this session actually got.

In [ ]:
!nvidia-smi

## 1. Install Ollama

`llama-cpp-python`'s pip install tries to build llama.cpp from source on
Kaggle and fails (`ggml-org/llama.cpp` doesn't publish a prebuilt CUDA binary
for Linux, only Windows -- confirmed directly against its GitHub releases).
Ollama bundles the same GGUF/llama.cpp engine as a static binary with CUDA
support already compiled in (it even bundles its own CUDA 12 and CUDA 13
runtime libraries, so it doesn't depend on Kaggle's installed CUDA toolkit
version matching).

This skips Ollama's own `curl | sh` installer script deliberately: piping a
~1.4GB download through a shell script inside a notebook cell was seen to
fail silently (`ollama: command not found` afterwards, with no visible
error). Downloading and extracting the release tarball directly, and calling
the binary by its absolute path rather than trusting `PATH`, avoids that
failure mode entirely (ADR-0002's 2026-08-31 update).


In [ ]:
!pip install -q zstandard

import tarfile

import zstandard

!curl -fL --retry 3 -o /tmp/ollama.tar.zst https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst

with (
    open("/tmp/ollama.tar.zst", "rb") as f,
    zstandard.ZstdDecompressor().stream_reader(f) as reader,
    tarfile.open(fileobj=reader, mode="r|") as tar,
):
    tar.extractall("/usr/local", filter="data")

OLLAMA_BIN = "/usr/local/bin/ollama"
!{OLLAMA_BIN} --version

## 2. Start Ollama and pull the model

Ollama pulls GGUF checkpoints directly from Hugging Face -- no separate
download step, no manual file path. Verify this repo id and quant tag still
exist at https://huggingface.co/unsloth/Qwen3-30B-A3B-GGUF before relying on
this cell; swap the tag if `Q4_K_M` isn't listed.


In [ ]:
import subprocess
import time

OLLAMA_MODEL = "hf.co/unsloth/Qwen3-30B-A3B-GGUF:Q4_K_M"

# Runs for the rest of the notebook's life; Kaggle's cell finishes once the
# server is listening, it doesn't wait for the process to exit.
ollama_proc = subprocess.Popen([OLLAMA_BIN, "serve"])
time.sleep(5)  # give the server a moment to bind before the first request

!{OLLAMA_BIN} pull {OLLAMA_MODEL}

## 3. Smoke-test

This is the step that answers the project's biggest open risk: does this
runtime actually support Qwen3-30B-A3B's MoE architecture? If the request
below errors or returns garbage, stop here -- don't proceed to the
throughput benchmark or the harness wiring below.


In [ ]:
import requests

OLLAMA_BASE_URL = "http://127.0.0.1:11434"

smoke = requests.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={
        "model": OLLAMA_MODEL,
        "messages": [{"role": "user", "content": "Reply with exactly the word: OK"}],
        "stream": False,
    },
    timeout=120,
)
smoke.raise_for_status()
print(smoke.json()["message"]["content"])

## 4. Tokens/sec benchmark

Ollama's `/api/chat` response already reports `eval_count` (completion
tokens) and `eval_duration` (nanoseconds spent generating them), so this
doesn't need to time the request by hand.


In [ ]:
BENCH_PROMPT = (
    "Describe, in detail, a strategy for exploring an unfamiliar grid-based "
    "puzzle game one action at a time."
)
BENCH_MAX_TOKENS = 512

result = requests.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={
        "model": OLLAMA_MODEL,
        "messages": [{"role": "user", "content": BENCH_PROMPT}],
        "stream": False,
        "options": {"num_predict": BENCH_MAX_TOKENS},
    },
    timeout=300,
)
result.raise_for_status()
payload = result.json()

completion_tokens = payload["eval_count"]
eval_seconds = payload["eval_duration"] / 1e9
tokens_per_second = completion_tokens / eval_seconds if eval_seconds > 0 else None
print(
    f"completion_tokens={completion_tokens} eval_seconds={eval_seconds:.2f}s tokens/sec={tokens_per_second:.2f}"
)

## 5. Confirm the OpenAI-compatible endpoint

`ollama serve` (already running since step 2) also exposes an
OpenAI-compatible `/v1` surface on the same port -- this is what
`arc-agi-3-benchmarking` will actually call (`configs/model_configs.local.yaml`).


In [ ]:
import os

os.environ["LOCAL_LLAMACPP_API_KEY"] = (
    "not-needed"  # Ollama's OpenAI-compat endpoint doesn't check this, but arc-agi-3-benchmarking's client requires a non-empty string
)

In [ ]:
resp = requests.get(f"{OLLAMA_BASE_URL}/v1/models", timeout=10)
print(resp.status_code, resp.json())

## 6. Wire into arc-agi-3-benchmarking

Clones the official benchmarking harness fresh into the notebook, appends
this project's local-model config entry (`configs/model_configs.local.yaml`)
into its `model_configs.yaml`, and installs its dependencies.


In [ ]:
!git clone --depth 1 https://github.com/arcprize/arc-agi-3-benchmarking.git /kaggle/working/arc-agi-3-benchmarking
!pip install -q -r /kaggle/working/arc-agi-3-benchmarking/requirements.txt || pip install -q anthropic google-genai openai pydantic requests python-dotenv

In [ ]:
import yaml

LOCAL_CONFIG_PATH = (
    "configs/model_configs.local.yaml"  # this repo, checked out alongside the notebook
)
TARGET_CONFIG_PATH = (
    "/kaggle/working/arc-agi-3-benchmarking/benchmarking/model_configs.yaml"
)

with open(LOCAL_CONFIG_PATH) as f:
    local_entries = yaml.safe_load(f)

with open(TARGET_CONFIG_PATH) as f:
    target_entries = yaml.safe_load(f) or []

existing_ids = {entry["id"] for entry in target_entries}
for entry in local_entries:
    if entry["id"] not in existing_ids:
        target_entries.append(entry)

with open(TARGET_CONFIG_PATH, "w") as f:
    yaml.safe_dump(target_entries, f, sort_keys=False)

print(f"Config ids now available: {[entry['id'] for entry in target_entries]}")

## 7. Run the baseline against public games

Set `ARC_API_KEY` via Kaggle's Secrets (Add-ons -> Secrets), not a literal
string in this cell. Pick a small handful of public game-id prefixes for a
first pass -- widen once this works.


In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["ARC_API_KEY"] = UserSecretsClient().get_secret("ARC_API_KEY")

GAME_PREFIXES = (
    "ls20,vc33"  # adjust to real available game-id prefixes; see --list-games
)

In [ ]:
%cd /kaggle/working/arc-agi-3-benchmarking
!python main.py --game={GAME_PREFIXES} --config=qwen3-30b-a3b-local-gguf

## 8. Summarize: RHAE + tokens/sec

Uses this project's `benchmark_report` module against the run's `logs.log`.

In [ ]:
import sys

sys.path.insert(
    0, "/kaggle/working/solve-arc-agi-3/src"
)  # if this repo is checked out on Kaggle
# If this import fails, this repo isn't checked out in this session: copy
# src/solve_arc_agi_3/benchmark_report.py's contents into a cell above and
# re-run, or `pip install` this repo once it has a git remote.
from solve_arc_agi_3.benchmark_report import summarize

with open("logs.log") as f:
    log_text = f.read()

report = summarize(log_text=log_text, tokens_per_second=tokens_per_second)
print("tokens/sec:", report.tokens_per_second)
print("mean RHAE:", report.mean_rhae)
for g in report.games:
    print(f"  {g.game_id}: rhae={g.rhae} actions={g.actions}")

## 9. Save the result

Append this run's row to a results file back in the repo, so all GPU
options end up in one place for the demo table in `docs/SLICES.md` V1.
Fill in which GPU option this session used (from cell 0's `nvidia-smi`
output) before running this cell.


In [ ]:
import csv
import datetime
from pathlib import Path

GPU_OPTION = "<fill in: T4x2 / P100 / RTX6000>"

RESULTS_PATH = Path("docs/research/phase0-results/results.csv")
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

is_new = not RESULTS_PATH.exists()
with RESULTS_PATH.open("a", newline="") as f:
    writer = csv.writer(f)
    if is_new:
        writer.writerow(["timestamp", "gpu", "tokens_per_second", "mean_rhae", "games"])
    writer.writerow(
        [
            datetime.datetime.now(tz=datetime.UTC).isoformat(),
            GPU_OPTION,
            report.tokens_per_second,
            report.mean_rhae,
            ";".join(g.game_id or "?" for g in report.games),
        ]
    )

print(f"Appended to {RESULTS_PATH}")